# Lab 06｜怎样借助旧经历，开始一场新的调查？

这是专栏《自进化 Agent 实战》06 讲《会话分支：怎样借助旧经历，开始一场新的调查？》的配套练习。练习使用构造的事故材料，目标是掌握处理真实场景的能力。

## 题面

一条已经用了几个月的会话，一直围绕新支付平台积累：熟悉它的灰度发布、服务网格和流量异常处理。今天你切到一套多年前的历史模块目录，把另一起告警直接发到这条会话，发现它总在按新平台的结构找东西，查不到预期实例就怀疑上报缺失。另一种做法是直接翻出一年前那起旧事故会话接着用，可那样今天的新现场会续写在旧结论后面。你要的是一条新会话，有边界地继承旧调查里仍然有用的背景，又不把旧事故的现场和结论当成今天的起点。你会怎么划定继承的位置、建立新时间线，再让它重新确认这些入口今天是否仍然有效？

## 参考答案

先看清问题出在任务边界。旧会话里积累的是新平台的拓扑、日志路径和故障模式，工作目录虽然切到历史模块，会话里的上下文还在拿另一套系统解释眼前的故障——上下文很充足，只是归属错了。直接接着用旧事故会话也不对：那会把今天的工具结果续写到旧结论后面，两场事故共用一条时间线，旧结论反而成了新排查的起点。完全开一条空白会话最干净，却又得重新摸一遍部署、日志和监控入口。

所以需要的是分岔：在旧会话上选一个位置，只继承到这个位置为止的历史，之后的旧事故现场不进入新分支。这个位置应当选在"只建立了模块背景、还没进入上一场事故判断"的那一轮。新分支拿到的是历史模块怎么部署、日志和监控从哪里取，而不是"连接池被调低过"这类只属于上一场事故的观察。

分岔后要为新任务重新声明边界：继承来的内容只用来认门，今天的入口必须重新确认。新会话会把这些背景当线索，再用工具调用核对它们今天是否还有效，然后读取今天的发布、日志、指标，形成当前事故自己的证据链。三条会话由此各自保存三项任务的状态——可以有来源关系、共享一段历史前缀，但各自的目标、现场观察、判断和待办必须分开。

## 准备

使用 Python 3.12 内核；在线运行时依赖自动补装，未设置 `GLM_API_KEY` 时弹出隐藏输入框。`lab.codex` 是高层 SDK 客户端；`story.fork_client` 是准备好的底层 `CodexClient`，用于高层对象尚未提供的选轮分岔接口。
下面直接展示创建会话、运行任务、按 `lastTurnId` 分岔以及读取父子历史。`lab.watch()` 只做超时保护，`lab.record_result()` 只保存结果。离线只回放材料，最后一格关闭连接。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook06_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook06_setup import prepare
story = prepare()

## Agent 为什么一直在排查另一套系统？

先给会话积累新支付平台背景，再切到历史模块目录提交告警；两套目录与实例、时间、数值均为构造材料。
跑完对照当前目录、实际查询入口与回答，观察会话上下文的归属怎样干扰新判断；litellm 代理层转发模型请求，不改写任务历史。
离线只比较材料入口，不产生模型回答。

In [ ]:
lab, platform, legacy = story.lab, story.platform, story.legacy
if lab:
    from openai_codex import ApprovalMode, Codex, Sandbox
    from openai_codex.api import Thread
codex = lab.codex if lab else None
platform_thread = codex.thread_start(cwd=str(platform), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None
if lab:
    turn = platform_thread.turn(story.platform_context + "只确认背景，不调用工具。")
    with lab.watch(turn):
        background = turn.run()
    lab.record_result(turn, background, show_result=False)
wrong_start = len(lab.requests) if lab else 0


In [ ]:
if lab:
    turn = platform_thread.turn(story.alert, cwd=str(legacy), sandbox=Sandbox.full_access)
    with lab.watch(turn):
        wrong = turn.run()
    lab.record_result(turn, wrong, show_result=False)
    probes = []
else:
    wrong, probes = None, story.replay_wrong()
wrong_calls = lab.requests[wrong_start:] if lab else []
story.show_wrong(wrong, wrong_calls, probes)


## Fork 怎样为新的事故建立独立的 Thread？

读取旧事故的构造记录，选择只建立模块背景、还没进入事故判断的第一轮，按包含边界继承历史。
跑完对照保留的前缀与排除的后续历史；在线核对新编号、来源和已结束轮次，离线只对材料切片。
历史边界决定模型继承什么；引用式继承共享选定前缀，新 Thread 单独追加后续记录。

In [ ]:
old_records, boundary = story.old_turns, 1
parent = codex.thread_start(cwd=str(legacy), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None
old_turns = []
if lab:
    for record in old_records:
        handle = parent.turn(record["text"] + "仅确认这条历史记录，不调用工具。")
        with lab.watch(handle):
            result = handle.run()
        lab.record_result(handle, result, show_result=False)
        old_turns.append((record, result))
parent_before = parent.read(include_turns=True).model_dump(mode="json", by_alias=True) if lab else None
story.show_history(parent_before, old_turns)


In [ ]:
selected = old_records[:boundary]
if lab:
    fork_client = story.fork_client
    response = fork_client.thread_fork(parent.id, {
        "lastTurnId": old_turns[boundary - 1][1].id, "cwd": str(legacy),
        "sandbox": "danger-full-access", "approvalPolicy": "never"})
    branch = Thread(fork_client, response.thread.id)
    branch_before = branch.read(include_turns=True).model_dump(mode="json", by_alias=True)
else:
    branch, branch_before = None, None
story.show_fork(parent, branch, parent_before, branch_before, boundary, selected)


## 新 Thread 怎样接回模块背景，重新调查当前告警？

在新会话声明继承只用来认门，重新确认入口，再读取今天的发布、日志、指标、配置与代码。
跑完对照当前材料和调查结果，检查新时间线独立、父记录未变；隔离重放只用内存计数，不操作真实服务。
这里起作用的是新时间线与旧前缀分离，当前证据链独立；离线实际查询材料，不证明模型完成调查。

In [ ]:
task = ("当前任务是 INC-2026-0917。继承只用于了解模块背景，不作为当前结论。"
        "重新确认入口是否有效：在本目录使用 Python 执行 query.py --list，再查询各入口。"
        "读取今天的发布、应用日志、数据库指标、连接池配置和配置指向的代码，按 request_id 关联日志。"
        "运行本目录的 replay_retry.py 做隔离重放；只调查与报告，不实施变更，不查询目录外或网络。"
        "中文报告区分当前已确认事实、推断和待验证项，注明查询入口、实例版本及重放的适用范围。")
current_start = len(lab.requests) if lab else 0
result = None
if lab:
    turn = branch.turn(task, sandbox=Sandbox.full_access)
    with lab.watch(turn):
        result = turn.run()
    lab.record_result(turn, result, show_result=False)
replay = None if lab else story.replay_current()


In [ ]:
current_calls = lab.requests[current_start:] if lab else []
after = story.snapshot(story.work)
print("输入材料：", "未改变" if after == story.input_snapshot else "有变化")
story.show_current(result, current_calls, replay, parent, branch, parent_before, after)